# Пайплайн

In [ ]:
'''
Простой пайплайн: признаки -> подбор параметров -> кросс-валидация -> смесь -> отправки.
Запускается целиком, без переключателей. Ускоритель T4 (первая строка вывода должна быть GPU: True), около 2,5–3 часов.

Что известно из исследований (ноутбуки _data_overview, _boosting_baseline, _research и другие):
- 222 признака на полной двухлетней истории: 0.722 на кросс-валидации, 0.718 на Public; шум Public ± 0.017;
- главный сигнал — ритм заказов и повторяемость покупок;
- признаки по отдельным заказам (заказы одного дня не склеиваются) дают +0.011, структура интервалов,
  заказы по интервалам и календарь — ещё +0.002…+0.004 каждая (ноутбук _feature_search_orders);
- локальная кросс-валидация совпадает с Public, поэтому решения принимаются по ней.

Шаги:
1. признаки: 222 прежних + около 260 новых по заказам, интервалам и календарю;
   новые проходят отбор — остаются те, чья важность выше, чем на перемешанном таргете;
2. подбор параметров Optuna для LightGBM, XGBoost и CatBoost на трёх фолдах;
3. кросс-валидация на 5 фолдах: исходные параметры против подобранных, берём лучшие;
4. усреднение по зёрнам; прогноз на тест — среднее по моделям всех фолдов и зёрен;
5. смесь моделей по рангам: равная и с весами, подобранными по прогнозам вне фолда;
6. файлы: submission.csv (смесь с весами), submission_equal.csv (равная смесь), по одной на модель;
7. разбор результата: где модель ошибается и какие признаки важны; признаки сохраняются в features_v2.parquet,
   чтобы следующие эксперименты не пересчитывали их.
'''
import itertools
import json
import os
import re
import subprocess
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import xgboost as xgb
from catboost import CatBoostClassifier
from IPython.display import display
from scipy.special import ndtr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("LAMODA_OUT_DIR", "."))

SEED = 42
GAP_DAYS = 14
TARGET_MONTHS = 3
WINDOWS = [7, 14, 30, 60, 90, 180, 365]
N_FOLDS = 5
N_SEEDS = 3                                    # зёрен на модель в итоговой кросс-валидации
TUNE_FOLDS = 3
N_TRIALS = {"lgb": 30, "xgb": 25, "cb": 20}    # попыток подбора на модель
TUNE_TIMEOUT = 35 * 60                         # и не дольше стольких секунд на модель
EARLY_STOP = 150
try:
    GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    GPU = False
print("GPU:", GPU)
T0 = time.time()


def fast_ap(target, score):
    order = np.argsort(-np.asarray(score), kind="mergesort")
    t = np.asarray(target)[order]
    k = int(np.ceil(len(t) * 0.20))
    top = t[:k]
    return float((np.cumsum(top) / np.arange(1, k + 1) * top).sum() / max(min(k, int(t.sum())), 1))

## Данные и признаки

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["scoring_date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["scoring_date"])
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
orders = pd.read_parquet(DATA_DIR / "orders.parquet")
products = pd.read_parquet(DATA_DIR / "products.parquet")
orders["order_date"] = pd.to_datetime(orders["order_date"])

cust = pd.concat([train.assign(part="train"), test.assign(part="test", target=np.nan)], ignore_index=True)
cust["slice"] = cust["scoring_date"].dt.strftime("%Y-%m-%d")
cust["cutoff"] = cust["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
cust["win_len"] = cust["scoring_date"].map(lambda d: (d + pd.DateOffset(months=TARGET_MONTHS) - d).days)
assert cust["customer_id"].is_unique

lines = orders.merge(products, on="product_id", how="left", validate="m:1").merge(
    cust[["customer_id", "cutoff"]], on="customer_id", how="inner", validate="m:1"
)
lines["d"] = (lines["cutoff"] - lines["order_date"]).dt.days
assert len(lines) == len(orders) and lines["d"].between(0, 732).all()


def clean(name):
    return re.sub(r"\W+", "_", str(name)).strip("_").lower()


def build_features(lines, win_len):
    lines = lines.copy()
    lines["has_disc"] = (lines["discount_percent"] > 0).astype("float32")
    lines["has_loy"] = (lines["loyalty_percent"] > 0).astype("float32")
    lines["weekend"] = (lines["order_date"].dt.dayofweek >= 5).astype("float32")

    # --- уровень заказа, самые свежие заказы идут первыми
    ot = (
        lines.groupby(["customer_id", "order_id"], sort=False)
        .agg(
            d=("d", "first"),
            order_sum=("paid_price", "sum"),
            n_items=("paid_price", "size"),
            disc=("discount_percent", "mean"),
            loy=("loyalty_percent", "max"),
            weekend=("weekend", "first"),
        )
        .reset_index()
        .sort_values(["customer_id", "d", "order_id"])
        .reset_index(drop=True)
    )
    cid = ot["customer_id"]
    f = ot.groupby("customer_id").agg(
        recency=("d", "min"),
        tenure=("d", "max"),
        d_mean=("d", "mean"),
        d_std=("d", "std"),
        n_orders=("order_id", "size"),
        n_items=("n_items", "sum"),
        sum_paid=("order_sum", "sum"),
        order_sum_mean=("order_sum", "mean"),
        order_sum_median=("order_sum", "median"),
        order_sum_max=("order_sum", "max"),
        order_sum_std=("order_sum", "std"),
        items_per_order=("n_items", "mean"),
        items_max=("n_items", "max"),
        order_disc_mean=("disc", "mean"),
        order_loy_mean=("loy", "mean"),
        order_loy_max=("loy", "max"),
        weekend_share=("weekend", "mean"),
    )

    last = ot.drop_duplicates("customer_id").set_index("customer_id")
    first = ot.drop_duplicates("customer_id", keep="last").set_index("customer_id")
    f["last_order_sum"] = last["order_sum"]
    f["last_n_items"] = last["n_items"]
    f["last_disc"] = last["disc"]
    f["last_loy"] = last["loy"]
    f["first_loy"] = first["loy"]
    f["first_order_sum"] = first["order_sum"]
    f["loy_change"] = f["last_loy"] - f["first_loy"]
    f["last_sum_to_mean"] = f["last_order_sum"] / f["order_sum_mean"]

    # --- окна по времени
    for w in WINDOWS:
        m = ot["d"] <= w
        f[f"n_orders_{w}"] = m.groupby(cid).sum()
        f[f"sum_{w}"] = (ot["order_sum"] * m).groupby(cid).sum()
        f[f"n_items_{w}"] = (ot["n_items"] * m).groupby(cid).sum()
    for w in [90, 365]:
        m = ot["d"] <= w
        f[f"loy_max_{w}"] = ot["loy"].where(m).groupby(cid).max()
        f[f"disc_mean_{w}"] = ot["disc"].where(m).groupby(cid).mean()
        f[f"aov_{w}"] = f[f"sum_{w}"] / f[f"n_orders_{w}"].replace(0, np.nan)
    f["n_orders_older_365"] = f["n_orders"] - f["n_orders_365"]

    # --- уровень дня: ритм заказов
    days = ot.groupby(["customer_id", "d"], as_index=False).agg(day_orders=("order_id", "size"), day_sum=("order_sum", "sum"))
    days["gap"] = days.groupby("customer_id")["d"].diff()
    days["rk"] = days.groupby("customer_id").cumcount()
    gd = days.groupby("customer_id")
    f["n_days"] = gd.size()
    f["max_day_orders"] = gd["day_orders"].max()
    f["multi_order_days"] = (days["day_orders"] > 1).groupby(days["customer_id"]).sum()
    f = f.join(gd["gap"].agg(gap_mean="mean", gap_median="median", gap_std="std", gap_min="min", gap_max="max"))
    for k in [1, 2, 4, 9]:
        f[f"d_day_{k + 1}"] = days.loc[days["rk"] == k].set_index("customer_id")["d"]  # давность (k+1)-го с конца дня с заказом
    f["gap_prev"] = days.loc[days["rk"] == 2].set_index("customer_id")["gap"]
    for name, period, limit in [("months_12", 30, 12), ("months_24", 30, 25), ("weeks_13", 7, 13), ("weeks_26", 7, 26)]:
        b = days.assign(b=days["d"] // period)
        f[f"active_{name}"] = b.loc[b["b"] < limit].drop_duplicates(["customer_id", "b"]).groupby("customer_id").size()
    f[[c for c in f.columns if c.startswith("active_")]] = f[[c for c in f.columns if c.startswith("active_")]].fillna(0)

    # --- попадание следующего заказа в целевое окно
    # Время считаем в днях после отсечения: окно — это (GAP_DAYS, GAP_DAYS + win_len].
    r = f["recency"]
    w0, w1 = GAP_DAYS, GAP_DAYS + win_len.reindex(f.index)
    dc = days["customer_id"]
    g, dr = days["gap"], dc.map(r)
    lo, hi = dr + GAP_DAYS, dr + GAP_DAYS + dc.map(win_len)
    n_gaps = g.notna().groupby(dc).sum()
    alive = (g > dr).groupby(dc).sum()  # сколько собственных интервалов клиента длиннее текущей паузы
    f["own_gaps_gt_recency"] = alive
    f["own_share_gaps_gt_recency"] = alive / n_gaps.replace(0, np.nan)
    f["own_p_hit"] = ((g > lo) & (g <= hi)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_hidden"] = ((g > dr) & (g <= lo)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_late"] = (g > hi).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_share_gaps_le_window"] = (g <= dc.map(win_len)).groupby(dc).sum() / n_gaps.replace(0, np.nan)

    wgt = 0.7 ** (days["rk"] - 1).clip(lower=0)
    f["gap_ewm"] = (g * wgt).groupby(dc).sum() / wgt.where(g.notna()).groupby(dc).sum()
    f["gap_last3"] = g.where(days["rk"].between(1, 3)).groupby(dc).mean()
    f["gap_last"] = days.loc[days["rk"] == 1].set_index("customer_id")["gap"]
    f["gap_last_to_median"] = f["gap_last"] / f["gap_median"]
    for name in ["gap_last", "gap_last3", "gap_ewm", "gap_median", "gap_mean", "gap_min", "gap_max"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r  # ожидаемый день следующего заказа; отрицательный — клиент «опаздывает»
        f[f"tnext_{short}"] = t
        f[f"tnext_{short}_dist"] = (w0 - t).clip(lower=0) + (t - w1).clip(lower=0)  # расстояние до окна, 0 — внутри
        if name in ["gap_last", "gap_ewm", "gap_median", "gap_mean"]:
            f[f"cycles_{short}"] = np.floor((r + w1) / gg) - np.floor((r + w0) / gg)  # сколько раз период укладывается в окно
            f[f"tmod_{short}"] = gg - (r % gg)  # до ближайшего кратного периоду дня, если клиент пропускал циклы

    # --- окно в единицах собственного периода и сглаженное попадание
    for name in ["gap_mean", "gap_median", "gap_last", "gap_last3"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r
        f[f"ulo_{short}"] = (r + w0) / gg  # следующий заказ попадает в окно, если ulo < 1 <= uhi
        f[f"uhi_{short}"] = (r + w1) / gg
        for sd in (10, 20):  # день следующего заказа известен с точностью ± sd дней
            f[f"phit{sd}_{short}"] = ndtr((w1 - t) / sd) - ndtr((w0 - t) / sd)
        f[f"phitrel_{short}"] = ndtr((w1 - t) / (0.25 * gg)) - ndtr((w0 - t) / (0.25 * gg))

    # --- дни с заказами по 30-дневным интервалам и в том же окне год и два назад
    mb = pd.crosstab(dc, (days["d"] // 30).clip(upper=24)).reindex(columns=range(24), fill_value=0)
    mb.columns = [f"mb_{i:02d}" for i in mb.columns]
    f = f.join(mb)
    dl = dc.map(win_len)
    for years, tag in [(1, "ya"), (2, "ya2")]:
        start, end = 365 * years - GAP_DAYS - dl, 365 * years - GAP_DAYS - 1
        f[f"{tag}_days"] = ((days["d"] >= start) & (days["d"] <= end)).groupby(dc).sum()
        f[f"{tag}_days_wide"] = ((days["d"] >= start - 30) & (days["d"] <= end + 30)).groupby(dc).sum()

    # --- производные
    f["formula"] = f["n_orders"] / (f["recency"] + 30)
    f["days_formula"] = f["n_days"] / (f["recency"] + 30)
    f["orders_per_30d"] = f["n_orders"] / (f["tenure"] + 30) * 30
    f["recency_to_gap_mean"] = f["recency"] / f["gap_mean"]
    f["recency_to_gap_median"] = f["recency"] / f["gap_median"]
    f["recency_minus_gap_max"] = f["recency"] - f["gap_max"]
    f["gap_cv"] = f["gap_std"] / f["gap_mean"]
    f["recency_to_tenure"] = f["recency"] / (f["tenure"] + 1)
    f["share_orders_30"] = f["n_orders_30"] / f["n_orders"]
    f["share_orders_90"] = f["n_orders_90"] / f["n_orders"]
    f["share_orders_365"] = f["n_orders_365"] / f["n_orders"]
    f["trend_90_vs_365"] = f["n_orders_90"] / (f["n_orders_365"] - f["n_orders_90"] + 1)
    f["trend_30_vs_90"] = f["n_orders_30"] / (f["n_orders_90"] - f["n_orders_30"] + 1)
    f["share_sum_90"] = f["sum_90"] / f["sum_paid"].replace(0, np.nan)

    # --- уровень позиции: цены, скидки, лояльность, состав покупок
    gl = lines.groupby("customer_id")
    f = f.join(
        gl.agg(
            price_mean=("paid_price", "mean"),
            price_median=("paid_price", "median"),
            price_max=("paid_price", "max"),
            price_min=("paid_price", "min"),
            disc_mean=("discount_percent", "mean"),
            disc_max=("discount_percent", "max"),
            disc_share=("has_disc", "mean"),
            loy_mean=("loyalty_percent", "mean"),
            loy_share=("has_loy", "mean"),
            n_loy_values=("loyalty_percent", "nunique"),
            n_products=("product_id", "nunique"),
            n_brands=("brand", "nunique"),
            n_cat=("category", "nunique"),
            n_cat2=("category2", "nunique"),
            n_gender=("gender_product", "nunique"),
        )
    )
    f["repeat_product_share"] = 1 - f["n_products"] / f["n_items"]
    f["brands_per_item"] = f["n_brands"] / f["n_items"]
    f["cat2_per_item"] = f["n_cat2"] / f["n_items"]

    for col, prefix in [("category", "cat"), ("gender_product", "gender")]:
        share = pd.crosstab(lines["customer_id"], lines[col], normalize="index")
        share.columns = [f"{prefix}_share_{clean(c)}" for c in share.columns]
        f = f.join(share)
        recent = lines.loc[lines["d"] <= 90]
        share = pd.crosstab(recent["customer_id"], recent[col], normalize="index")
        share.columns = [f"{prefix}_share90_{clean(c)}" for c in share.columns]
        f = f.join(share)

    for col in ["category2", "brand"]:
        top = (
            lines.groupby(["customer_id", col], observed=True).size().rename("n").reset_index()
            .sort_values(["customer_id", "n", col], ascending=[True, False, True])
            .drop_duplicates("customer_id").set_index("customer_id")
        )
        f[f"top_{col}_share"] = top["n"] / f["n_items"]
        f[f"top_{col}"] = top[col].astype(str)

    # --- ритм отдельно по категориям, подкатегориям и брендам
    for col in ["category", "category2", "brand"]:
        st = (
            lines[["customer_id", col, "d"]].drop_duplicates()
            .groupby(["customer_id", col], observed=True)["d"].agg(n="size", last="min", first="max")
        )
        st = st[st["n"] >= 2].reset_index()
        st["gap"] = (st["first"] - st["last"]) / (st["n"] - 1)
        st["t"] = st["gap"] - st["last"]
        hi = GAP_DAYS + st["customer_id"].map(win_len)
        st["dist"] = (GAP_DAYS - st["t"]).clip(lower=0) + (st["t"] - hi).clip(lower=0)
        st["hit"] = (st["dist"] == 0).astype(float)
        gs = st.groupby("customer_id")
        f[f"ser_{col}_n"] = gs.size()
        f[f"ser_{col}_hit"] = gs["hit"].sum()
        f[[f"ser_{col}_n", f"ser_{col}_hit"]] = f[[f"ser_{col}_n", f"ser_{col}_hit"]].fillna(0)
        f[f"ser_{col}_hit_share"] = gs["hit"].mean()
        f[f"ser_{col}_min_dist"] = gs["dist"].min()
        best = st.sort_values(["customer_id", "dist", "last"]).drop_duplicates("customer_id").set_index("customer_id")
        f[f"ser_{col}_best_t"] = best["t"]
        f[f"ser_{col}_best_gap"] = best["gap"]

    return f.replace([np.inf, -np.inf], np.nan)


feats = build_features(lines, cust.set_index("customer_id")["win_len"])
data = cust.merge(feats, left_on="customer_id", right_index=True, how="left", validate="1:1")
assert data["recency"].notna().all() and len(data) == len(cust)
data["top_category2"] = data["top_category2"].astype("category")
data["top_brand"] = data["top_brand"].astype(object).where(data["top_brand"].notna(), "none").astype(str)

## Признаки по отдельным заказам, интервалам и календарю

In [ ]:
cid = "customer_id"
info = cust.set_index(cid)
LO = pd.Series(float(GAP_DAYS), index=info.index)
HI = (GAP_DAYS + info["win_len"]).astype(float)
EST = ["mean", "median", "min", "max", "q25", "q75", "last", "last2", "last3", "ewm", "trim"]
HIST = {"вся": 10_000, "год": 365, "полгода": 182}


def month_idx(s):
    return s.dt.year * 12 + s.dt.month


def estimators(E):
    E = E.sort_values([cid, "t"]).reset_index(drop=True)
    g = E.groupby(cid)["t"]
    E["gap"] = g.diff()
    E["rk"] = E.groupby(cid).cumcount()
    gp = E[E["gap"].notna()]
    gg = gp.groupby(cid)["gap"]
    out = pd.DataFrame({"r": g.min(), "n": g.size()})
    out["mean"] = (g.max() - g.min()) / (out["n"] - 1).replace(0, np.nan)
    out["median"] = gg.median()
    out["min"] = gg.min()
    out["max"] = gg.max()
    out["q25"] = gg.quantile(0.25)
    out["q75"] = gg.quantile(0.75)
    out["last"] = gp[gp["rk"] == 1].set_index(cid)["gap"]
    out["last2"] = gp[gp["rk"] <= 2].groupby(cid)["gap"].mean()
    out["last3"] = gp[gp["rk"] <= 3].groupby(cid)["gap"].mean()
    w = 0.7 ** (gp["rk"] - 1)
    out["ewm"] = (gp["gap"] * w).groupby(gp[cid]).sum() / w.groupby(gp[cid]).sum()
    out["trim"] = (gg.sum() - gg.max()) / (gg.count() - 1).replace(0, np.nan)
    return out


def unit_feats(E, prefix):
    # «ожидаемый момент следующего заказа» по разным оценкам периода; E — (customer_id, d), d — дней до отсечения
    out = {}
    E = E.assign(t=E["d"].astype(float))
    for hname, H in HIST.items():
        est = estimators(E[E["d"] <= H][[cid, "t"]]).reindex(info.index)
        r = est["r"]
        key = f"{prefix}|{hname}"
        out[f"{key}|n"] = est["n"].fillna(0)
        out[f"{key}|r"] = r
        for e in EST:
            P = est[e].where(est[e] > 0)
            tn, tm = P - r, P - np.mod(r, P)
            pre = f"{key}|{e}"
            out[f"{pre}|t"] = tn
            out[f"{pre}|dist"] = (LO - tn).clip(lower=0) + (tn - HI).clip(lower=0)
            out[f"{pre}|in"] = ((tn > LO) & (tn <= HI)).astype("float32").where(P.notna())
            out[f"{pre}|tm_in"] = ((tm > LO) & (tm <= HI)).astype("float32").where(P.notna())
            out[f"{pre}|k_in"] = np.floor((r + HI) / P) - np.floor((r + LO) / P)
    return out


o = (
    lines.groupby([cid, "order_id"], sort=False)
    .agg(d=("d", "first"), date=("order_date", "first"), n_items=("paid_price", "size"))
    .reset_index()
    .sort_values([cid, "d", "order_id"])
    .reset_index(drop=True)
)
new = unit_feats(o[[cid, "d"]], "заказ")

# структура интервалов между заказами
o["gap"] = o.groupby(cid)["d"].diff()
gpo = o[o["gap"].notna()]
for name, cond in [("0", gpo["gap"] == 0), ("le1", gpo["gap"] <= 1), ("le3", gpo["gap"] <= 3), ("le7", gpo["gap"] <= 7), ("le14", gpo["gap"] <= 14),
                   ("le30", gpo["gap"] <= 30), ("gt60", gpo["gap"] > 60), ("gt90", gpo["gap"] > 90), ("gt180", gpo["gap"] > 180)]:
    cf = cond.astype(float)
    new[f"интервалы|доля {name}"] = cf.groupby(gpo[cid]).mean()
    new[f"интервалы|доля {name} за год"] = cf.where(gpo["d"] <= 365).groupby(gpo[cid]).mean()
    new[f"интервалы|число {name} за год"] = cf.where(gpo["d"] <= 365).groupby(gpo[cid]).sum()
dayc = o.groupby([cid, "d"]).agg(k=("order_id", "size"), items=("n_items", "sum")).reset_index()
for w in (90, 365):
    new[f"интервалы|дней с 2+ заказами за {w}"] = ((dayc["k"] >= 2) & (dayc["d"] <= w)).groupby(dayc[cid]).sum()
last_day = dayc.sort_values([cid, "d"]).drop_duplicates(cid).set_index(cid)
new["интервалы|заказов в последний день"] = last_day["k"]
new["интервалы|позиций в последний день"] = last_day["items"]
new["интервалы|давность последнего дня с 2+ заказами"] = dayc[dayc["k"] >= 2].groupby(cid)["d"].min()
r_o = o.groupby(cid)["d"].min()
qs = gpo.groupby(cid)["gap"].quantile([0.25, 0.5, 0.75, 0.9]).unstack()
for q in qs.columns:
    new[f"интервалы|давность / квантиль {q}"] = r_o / qs[q].replace(0, np.nan)
    new[f"интервалы|давность - квантиль {q}"] = r_o - qs[q]
new["интервалы|разброс"] = gpo.groupby(cid)["gap"].std() / gpo.groupby(cid)["gap"].mean().replace(0, np.nan)
pos = gpo[gpo["gap"] > 0].groupby(cid)["gap"]
new["интервалы|средний ненулевой"] = pos.mean()
new["интервалы|давность - средний ненулевой"] = r_o - pos.mean()

# заказы и позиции по непересекающимся интервалам до отсечения
BINS = [0, 7, 14, 30, 60, 90, 120, 180, 270, 365, 450, 540, 733]
labels = [f"{BINS[i]}-{BINS[i + 1]}" for i in range(len(BINS) - 1)]
t_o = pd.crosstab(o[cid], pd.cut(o["d"], BINS, right=False, labels=labels)).reindex(columns=labels, fill_value=0)
t_l = pd.crosstab(lines[cid], pd.cut(lines["d"], BINS, right=False, labels=labels)).reindex(columns=labels, fill_value=0)
for lab in labels:
    new[f"по интервалам|заказов {lab}"] = t_o[lab]
    new[f"по интервалам|позиций {lab}"] = t_l[lab]
    new[f"по интервалам|доля заказов {lab}"] = t_o[lab] / t_o.sum(axis=1)

# календарь
mi = month_idx(o["date"])
rel = o[cid].map(month_idx(info["scoring_date"])) - mi
last_o = o.drop_duplicates(cid).set_index(cid)
new["календарь|день месяца последнего заказа"] = last_o["date"].dt.day
new["календарь|день недели последнего заказа"] = last_o["date"].dt.dayofweek
new["календарь|месяц последнего заказа"] = last_o["date"].dt.month
new["календарь|доля заказов 1–10 числа"] = (o["date"].dt.day <= 10).groupby(o[cid]).mean()
new["календарь|доля заказов 21+ числа"] = (o["date"].dt.day >= 21).groupby(o[cid]).mean()
new["календарь|заказов в месяцы окна год назад"] = rel.between(10, 12).groupby(o[cid]).sum()
new["календарь|заказов в месяцы окна два года назад"] = rel.between(22, 24).groupby(o[cid]).sum()
new["календарь|заказов в 3 месяца перед окном год назад"] = rel.between(13, 15).groupby(o[cid]).sum()
new["календарь|заказов в 3 месяца после окна год назад"] = rel.between(7, 9).groupby(o[cid]).sum()
new["календарь|активных месяцев"] = mi.groupby(o[cid]).nunique()
new["календарь|активных месяцев за год"] = mi.where(rel <= 12).groupby(o[cid]).nunique()

NEW = pd.DataFrame(new).reindex(info.index).astype("float32")
NAMES = {f"n{i}": c for i, c in enumerate(NEW.columns)}       # короткие имена для моделей -> понятные имена
NEW.columns = list(NAMES)
data = data.join(NEW, on=cid)
BASE = [c for c in feats.columns if c != "top_brand"]
print(f"новых признаков: {NEW.shape[1]} | {(time.time() - T0) / 60:.1f} мин")

## Отбор новых признаков

In [ ]:
# Новый признак остаётся, если его важность выше 90-го процентиля важности на перемешанном таргете
trd = data[data["part"].eq("train")]
Xs, ys = trd[BASE + list(NAMES)], trd["target"].astype(int).to_numpy()
sel = dict(objective="binary", learning_rate=0.05, n_estimators=300, num_leaves=31, min_child_samples=100,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.5, n_jobs=-1, verbose=-1)


def gain(target, seed):
    m = lgb.LGBMClassifier(**sel, random_state=seed).fit(Xs, target)
    return pd.Series(m.booster_.feature_importance("gain"), index=Xs.columns)


rng = np.random.default_rng(SEED)
real = pd.concat([gain(ys, s) for s in range(3)], axis=1).mean(axis=1)
null = pd.concat([gain(rng.permutation(ys), s) for s in range(6)], axis=1)
ratio = real / (null.quantile(0.9, axis=1) + 1e-9)
KEEP_NEW = [c for c in NAMES if ratio[c] > 1.0]
FEATURES = BASE + KEEP_NEW
print(f"новых признаков выше шума: {len(KEEP_NEW)} из {len(NAMES)} | всего признаков: {len(FEATURES)} | {(time.time() - T0) / 60:.1f} мин")
print("Самые важные из новых:")
display((real[KEEP_NEW] / real.sum()).sort_values(ascending=False).head(20).rename(index=NAMES).to_frame("доля важности"))

In [ ]:
tr = data[data["part"].eq("train")].reset_index(drop=True)
te = data[data["part"].eq("test")].set_index("customer_id").loc[sample["customer_id"]].reset_index()
y = tr["target"].astype(int).to_numpy()
sl = tr["slice"].to_numpy()


def frame(kind, df):
    # CatBoost получает ещё бренд и берёт категориальные признаки строками; остальным хватает подкатегории
    if kind != "cb":
        return df[FEATURES]
    out = df[FEATURES + ["top_brand"]].copy()
    out["top_category2"] = out["top_category2"].astype(str)
    return out


X = {k: frame(k, tr) for k in ["lgb", "xgb", "cb"]}
X_TE = {k: frame(k, te) for k in ["lgb", "xgb", "cb"]}
print("признаков:", len(FEATURES), "| train:", len(tr), "| test:", len(te), f"| {(time.time() - T0) / 60:.1f} мин")


def score(rows, pred):
    # среднее AP@20% по срезам; pred идёт в том же порядке, что rows
    return float(np.mean([fast_ap(y[rows][sl[rows] == s], pred[sl[rows] == s]) for s in np.unique(sl[rows])]))

## Модели

In [ ]:
FIXED = {
    "lgb": dict(objective="binary", learning_rate=0.03, n_estimators=5000, subsample_freq=1, cat_smooth=50, n_jobs=-1, verbose=-1),
    "xgb": dict(learning_rate=0.03, n_estimators=5000, tree_method="hist", device="cuda" if GPU else "cpu", enable_categorical=True,
                eval_metric="logloss", early_stopping_rounds=EARLY_STOP, n_jobs=-1, verbosity=0),
    "cb": dict(loss_function="Logloss", learning_rate=0.05, iterations=5000, bootstrap_type="Bayesian", task_type="GPU" if GPU else "CPU",
               cat_features=["top_category2", "top_brand"], thread_count=-1, verbose=0),
}
DEFAULT = {
    "lgb": dict(num_leaves=31, min_child_samples=100, colsample_bytree=0.5, subsample=0.8, reg_lambda=5.0, reg_alpha=1e-3, min_split_gain=0.0, max_bin=255),
    "xgb": dict(max_depth=6, min_child_weight=20.0, subsample=0.8, colsample_bytree=0.5, reg_lambda=5.0, reg_alpha=1e-3, gamma=0.0, max_bin=256),
    "cb": dict(depth=6, l2_leaf_reg=5.0, random_strength=1.0, bagging_temperature=1.0, border_count=128),
}
SPACE = {
    "lgb": lambda t: dict(
        num_leaves=t.suggest_int("num_leaves", 8, 96, log=True), min_child_samples=t.suggest_int("min_child_samples", 20, 400, log=True),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.2, 0.9), subsample=t.suggest_float("subsample", 0.5, 1.0),
        reg_lambda=t.suggest_float("reg_lambda", 0.1, 50, log=True), reg_alpha=t.suggest_float("reg_alpha", 1e-3, 10, log=True),
        min_split_gain=t.suggest_float("min_split_gain", 0.0, 0.5), max_bin=t.suggest_categorical("max_bin", [63, 127, 255])),
    "xgb": lambda t: dict(
        max_depth=t.suggest_int("max_depth", 3, 9), min_child_weight=t.suggest_float("min_child_weight", 1, 200, log=True),
        subsample=t.suggest_float("subsample", 0.5, 1.0), colsample_bytree=t.suggest_float("colsample_bytree", 0.2, 0.9),
        reg_lambda=t.suggest_float("reg_lambda", 0.1, 50, log=True), reg_alpha=t.suggest_float("reg_alpha", 1e-3, 10, log=True),
        gamma=t.suggest_float("gamma", 0.0, 2.0), max_bin=t.suggest_categorical("max_bin", [64, 128, 256])),
    "cb": lambda t: dict(
        depth=t.suggest_int("depth", 4, 9), l2_leaf_reg=t.suggest_float("l2_leaf_reg", 1, 30, log=True),
        random_strength=t.suggest_float("random_strength", 0.1, 5, log=True), bagging_temperature=t.suggest_float("bagging_temperature", 0.0, 2.0),
        border_count=t.suggest_categorical("border_count", [64, 128, 254])),
}


def fit(kind, params, a, b, seed):
    # обучение на строках a с ранней остановкой по строкам b; возвращает функцию прогноза
    Xa, Xb = X[kind].iloc[a], X[kind].iloc[b]
    if kind == "lgb":
        m = lgb.LGBMClassifier(**FIXED[kind], **params, random_state=seed).fit(Xa, y[a], eval_set=[(Xb, y[b])], callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    elif kind == "xgb":
        m = xgb.XGBClassifier(**FIXED[kind], **params, random_state=seed).fit(Xa, y[a], eval_set=[(Xb, y[b])], verbose=False)
    else:
        m = CatBoostClassifier(**FIXED[kind], **params, random_seed=seed).fit(Xa, y[a], eval_set=(Xb, y[b]), early_stopping_rounds=EARLY_STOP)
    return lambda Z: m.predict_proba(Z)[:, 1]


strata = tr["slice"] + "_" + tr["target"].astype(int).astype(str)
TUNE_SPLIT = list(StratifiedKFold(TUNE_FOLDS, shuffle=True, random_state=SEED + 1000).split(tr, strata))


def cv_run(kind, params, seeds):
    # прогнозы вне фолда на train и прогноз на тест, усреднённые по зёрнам
    oof, test = np.zeros(len(tr)), np.zeros(len(te))
    for seed in seeds:
        for a, b in StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + seed).split(tr, strata):
            predict = fit(kind, params, a, b, SEED + seed)
            oof[b] += predict(X[kind].iloc[b]) / len(seeds)
            test += predict(X_TE[kind]) / (len(seeds) * N_FOLDS)
    return oof, test

## Подбор параметров и кросс-валидация

In [ ]:
ALL_ROWS = np.arange(len(tr))
RES, rows = {}, []
for kind in ["lgb", "xgb", "cb"]:
    t0 = time.time()

    def objective(trial):
        params = SPACE[kind](trial)
        vals = []
        for i, (a, b) in enumerate(TUNE_SPLIT):
            vals.append(score(b, fit(kind, params, a, b, SEED)(X[kind].iloc[b])))
            trial.report(float(np.mean(vals)), i)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(vals))

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED), pruner=optuna.pruners.MedianPruner(n_startup_trials=8))
    study.enqueue_trial(DEFAULT[kind])          # первая попытка — исходные параметры
    study.optimize(objective, n_trials=N_TRIALS[kind], timeout=TUNE_TIMEOUT)

    # честное сравнение на 5 фолдах с одним зерном: исходные против подобранных
    cand = {"исходные": DEFAULT[kind], "подобранные": study.best_params}
    first = {name: cv_run(kind, p, [0]) for name, p in cand.items()}
    aps = {name: score(ALL_ROWS, r[0]) for name, r in first.items()}
    best = max(aps, key=aps.get)
    rest = cv_run(kind, cand[best], list(range(1, N_SEEDS))) if N_SEEDS > 1 else first[best]
    w = 1 / N_SEEDS
    oof = first[best][0] * w + rest[0] * (1 - w) if N_SEEDS > 1 else first[best][0]
    test = first[best][1] * w + rest[1] * (1 - w) if N_SEEDS > 1 else first[best][1]
    RES[kind] = dict(params=cand[best], oof=oof, test=test)
    rows.append({"модель": kind, "попыток": len(study.trials), "исходные": aps["исходные"], "подобранные": aps["подобранные"], "взяты": best,
                 f"итог, {N_SEEDS} зерна": score(ALL_ROWS, oof), "минут": round((time.time() - t0) / 60, 1)})
    print(f"{kind}: исходные {aps['исходные']:.4f}, подобранные {aps['подобранные']:.4f} -> взяты {best}; итог {score(ALL_ROWS, oof):.4f}; {(time.time() - t0) / 60:.1f} мин; параметры: {cand[best]}")

print("AP@20% на кросс-валидации (среднее по срезам, прогнозы вне фолда):")
display(pd.DataFrame(rows).set_index("модель"))

## Смесь и отправки

In [ ]:
names = list(RES)
rank_oof = np.column_stack([pd.Series(RES[k]["oof"]).groupby(sl).rank(pct=True).to_numpy() for k in names])
rank_te = np.column_stack([pd.Series(RES[k]["test"]).rank(pct=True).to_numpy() for k in names])
equal = np.full(len(names), 1 / len(names))
grid = [np.array(w) / 20 for w in itertools.product(range(21), repeat=len(names)) if sum(w) == 20]
w_best = max(grid, key=lambda w: score(ALL_ROWS, rank_oof @ w))

table = {k: {"AP@20%": score(ALL_ROWS, rank_oof[:, i]), **{s: fast_ap(y[sl == s], rank_oof[sl == s, i]) for s in np.unique(sl)}} for i, k in enumerate(names)}
for name, w in [("равная смесь", equal), ("смесь с весами", w_best)]:
    table[name] = {"AP@20%": score(ALL_ROWS, rank_oof @ w), **{s: fast_ap(y[sl == s], (rank_oof @ w)[sl == s]) for s in np.unique(sl)}}
display(pd.DataFrame(table).T)
print("веса:", dict(zip(names, np.round(w_best, 2).tolist())))
print("Корреляция прогнозов моделей на тесте (Спирмен):")
display(pd.DataFrame(rank_te, columns=names).corr(method="spearman"))


def save(scores, fname):
    sub = pd.DataFrame({"customer_id": te["customer_id"].to_numpy(), "score": pd.Series(scores).rank(pct=True).to_numpy()})
    assert sub["customer_id"].tolist() == sample["customer_id"].tolist() and sub["score"].between(0, 1).all()
    sub.to_csv(OUT_DIR / fname, index=False)


save(rank_te @ w_best, "submission.csv")
save(rank_te @ equal, "submission_equal.csv")
for i, k in enumerate(names):
    save(rank_te[:, i], f"submission_{k}.csv")
pd.DataFrame({"customer_id": tr["customer_id"], "slice": sl, "target": y, **{k: RES[k]["oof"] for k in names}}).to_parquet(OUT_DIR / "oof.parquet", index=False)
pd.DataFrame({"customer_id": te["customer_id"], **{k: RES[k]["test"] for k in names}}).to_parquet(OUT_DIR / "test_pred.parquet", index=False)
json.dump({"weights": dict(zip(names, map(float, w_best))), "params": {k: RES[k]["params"] for k in names}}, open(OUT_DIR / "params.json", "w"), indent=1, default=float)
print(f"Готово: submission.csv и submission_equal.csv сохранены. Всего времени: {(time.time() - T0) / 60:.1f} мин")

## Разбор результата

In [ ]:
best = rank_oof @ w_best
res = pd.DataFrame({"slice": sl, "target": y, "n_days": tr["n_days"].to_numpy(), "score": best})
res["pct"] = res.groupby("slice")["score"].rank(pct=True, ascending=False)
res["n_bin"] = pd.cut(res["n_days"], [0, 1, 2, 3, 5, 9, 20, 10_000], labels=["1", "2", "3", "4-5", "6-9", "10-20", "21+"])
res["in_top"] = res["pct"] <= 0.20
print("Доля target = 1 по положению в рейтинге:")
display(res.groupby(pd.cut(res["pct"], [0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 1.0]), observed=True)["target"].agg(["mean", "size"]).T)
print("Верхние 20% по группам числа дней с заказами:")
display(pd.DataFrame({
    "клиентов": res.groupby("n_bin", observed=True).size(),
    "доля в топ-20%": res.groupby("n_bin", observed=True)["in_top"].mean(),
    "target в топе": res[res["in_top"]].groupby("n_bin", observed=True)["target"].mean(),
    "target вне топа": res[~res["in_top"]].groupby("n_bin", observed=True)["target"].mean(),
    "AUC внутри группы": res.groupby("n_bin", observed=True).apply(lambda g: roc_auc_score(g["target"], g["score"]) if g["target"].nunique() == 2 else np.nan),
}))

# важность признаков в LightGBM с итоговыми параметрами
a, b = next(iter(StratifiedKFold(5, shuffle=True, random_state=SEED).split(tr, strata)))
m = lgb.LGBMClassifier(**FIXED["lgb"], **RES["lgb"]["params"], random_state=SEED).fit(
    X["lgb"].iloc[a], y[a], eval_set=[(X["lgb"].iloc[b], y[b])], callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
imp = pd.Series(m.booster_.feature_importance("gain"), index=FEATURES)
imp = (imp / imp.sum()).sort_values(ascending=False)
print("Доля важности: прежние 222 признака и новые:", {"прежние": round(float(imp[BASE].sum()), 3), "новые": round(float(imp[KEEP_NEW].sum()), 3)})
print("Самые важные признаки:")
display(imp.head(30).rename(index=NAMES).to_frame("доля важности"))

# признаки и понятные имена — для следующих экспериментов
keep_cols = ["customer_id", "part", "slice", "target"] + FEATURES
data[keep_cols].to_parquet(OUT_DIR / "features_v2.parquet", index=False)
json.dump({"names": NAMES, "features": FEATURES}, open(OUT_DIR / "features_v2_names.json", "w"), ensure_ascii=False, indent=1)
print(f"Сохранено: features_v2.parquet, features_v2_names.json. Всего времени: {(time.time() - T0) / 60:.1f} мин")